# mv-PIKN: export the latent representation

This notebook performs **inference only**, using your existing 10,000-update checkpoint and processed EEG. It does not train a model or access the outer test subject. Start with Koopman, seed 20261028, inner_01.

It exports the 32-dimensional fused vector **z**, current unsmoothed class probabilities, labels, subject/file IDs, timestamps, validity, and readiness. It fits PCA on a uniform sample of **training** vectors and applies that fixed projector to validation. It makes class-colored and subject-colored plots plus an illustrative seizure trajectory.

The actual checkpoints and EEG remain in your Drive. They are not embedded here. The checkpoint identities from your supplied prediction exports are embedded so that the notebook can reject the wrong model version.

Choose a GPU runtime if available to accelerate inference. CPU also works. Reading the processed EEG can take time even though no training occurs.

## 1. Dependencies and Drive

This mounts Drive at the path used in your earlier notebooks. If your files are mounted elsewhere, adjust RUN in Section 2.

In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys
required = {'numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib','h5py':'h5py','sklearn':'scikit-learn','torch':'torch'}
missing = [package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',*missing])
try:
    from google.colab import drive
except ImportError:
    print('Outside Colab: set RUN to your local preprocessing directory.')
else:
    if not Path('/content/gdrive/MyDrive').is_dir():
        drive.mount('/content/gdrive')


## 2. Configuration

Normally leave the defaults and run all. The notebook searches RUN for a matching `last.pt`. If the short-training experiment is outside RUN, change SEARCH_ROOT to that experiment folder. Alternatively enter the exact checkpoint path in CHECKPOINT_OVERRIDE.

To compare all models, use `VARIANTS = ["classifier", "koopman", "physics"]`. Keep seed and inner fold consistent. Each model gets its own training-fitted PCA; their coordinates are not directly aligned across models.

If export is interrupted, rerun this cell for a new output directory and restart export. To change only the figures after a completed export, rerun Sections 6–7 without running Section 5 again.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

RUN = Path('/content/gdrive/MyDrive/mvPIKN_preprocessing/run_20261028T165011846018Z')
OUTER = 'outer_chb01'
SCOPE = 'inner_01'          # inner_01, inner_02, or inner_03
SEED = 20261028            # 20261028 or 20261029
VARIANTS = ['koopman']     # or ['classifier', 'koopman', 'physics']

SEARCH_ROOT = RUN
CHECKPOINT_OVERRIDE = {}   # example: {'koopman': '/content/gdrive/MyDrive/.../koopman/last.pt'}
BATCH_SIZE = 128           # lower to 32 if GPU memory is limited
PCA_FIT_WINDOWS = 50000    # uniform training-ready sample; includes excluded label -1
PLOT_WINDOWS = 12000       # uniform display sample per role; no synthetic points
PLOT_SEED = 20261101

STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
LOCAL_DIR = Path('/content' if Path('/content').is_dir() else Path.cwd()) / 'mvPIKN_latent_work' / STAMP
DRIVE_DEST = RUN / 'latent_exports_v1' / OUTER / str(SEED) / SCOPE / STAMP
LOCAL_DIR.mkdir(parents=True, exist_ok=False)
assert OUTER == 'outer_chb01' and SCOPE in ['inner_01','inner_02','inner_03']
assert SEED in [20261028,20261029] and len(VARIANTS) == len(set(VARIANTS)) and len(VARIANTS) > 0
assert set(VARIANTS) <= {'classifier','koopman','physics'}
assert BATCH_SIZE >= 1 and PCA_FIT_WINDOWS >= 3 and PLOT_WINDOWS >= 3
print('Local work:',LOCAL_DIR)
print('Final Drive destination:',DRIVE_DEST)


## 3. Embedded loader, model, and export functions

These are the existing FoldData and encoder definitions, plus the export/plotting code. The central extraction operation is `logits, z = model(raw, spec)`. No next-window observation enters it. Current-window z is not averaged with history; the history rule controls readiness only.

In [ ]:
# Existing data_loading.py
SOURCE = '"""Fold-scoped, causal CHB-MIT loading. No model fitting or alarm scoring."""\nfrom pathlib import Path\nimport json\nimport hashlib\nimport numpy as np\nimport pandas as pd\nimport h5py\n\nclass FoldData:\n    def __init__(self, run, outer=\'outer_chb01\', scope=\'inner_01\'):\n        self.run = Path(run)\n        self.cfg = json.loads((self.run/\'config.json\').read_text())\n        self.inventory = json.loads((self.run/\'locked_inventory.json\').read_text())\n        self.events = json.loads((self.run/\'locked_events.json\').read_text())\n        folds = json.loads((self.run/\'locked_folds.json\').read_text())\n        self.fold = next(f for f in folds if f[\'id\'] == outer)\n        self.outer, self.scope = outer, scope\n        if scope == \'outer_refit\':\n            self.roles = {\'train\': self.fold[\'development_subjects\'], \'test\': self.fold[\'test_subjects\']}\n        else:\n            inner = next(f for f in self.fold[\'inner_folds\'] if f[\'id\'] == scope)\n            self.roles = {\'train\': inner[\'train_subjects\'], \'validation\': inner[\'validation_subjects\']}\n        if set(self.roles[\'train\']) & set(self.fold[\'test_subjects\']):\n            raise ValueError(\'Held-out subject appears in training.\')\n        if len(set(sum(self.roles.values(), []))) != len(sum(self.roles.values(), [])):\n            raise ValueError(\'Roles overlap or contain duplicate subjects.\')\n        status = json.loads((self.run/\'status.json\').read_text())\n        if status.get(\'status\') not in [\'review_required\',\'preprocessing_complete\']:\n            raise ValueError(\'Preprocessing is incomplete.\')\n        blocking = [x for x in status.get(\'cohort_issues\', [])\n                    if x.get(\'reason\') != \'no_valid_preictal_input_after_QC\']\n        if blocking:\n            raise ValueError(f\'Unresolved preprocessing issues: {blocking}\')\n        if status.get(\'cohort_issues\'):\n            audit = json.loads((self.run/\'training_label_audit_status_v2_2.json\').read_text())\n            if audit.get(\'revision\') != \'2.2\':\n                raise ValueError(\'Run preprocessing Section 7B first.\')\n        self.scaler_path = self.run/\'folds\'/outer/scope/\'train_scaler.npz\'\n        with np.load(self.scaler_path, allow_pickle=False) as z:\n            if set(z[\'training_subjects\'].tolist()) != set(self.roles[\'train\']):\n                raise ValueError(\'Scaler training subjects do not match the fold.\')\n            if set(z[\'test_subjects\'].tolist()) != set(self.fold[\'test_subjects\']):\n                raise ValueError(\'Scaler outer test subjects do not match.\')\n            self.scaler = {k:z[k].copy() for k in [\'raw_mean\',\'raw_std\',\'feature_mean\',\'feature_std\']}\n        C = len(self.cfg[\'channels\'])\n        for k,shape in [(\'raw_mean\',(C,)),(\'raw_std\',(C,)),(\'feature_mean\',(C,5)),(\'feature_std\',(C,5))]:\n            v = self.scaler[k]\n            if v.shape != shape or not np.isfinite(v).all() or (\'std\' in k and np.any(v<=0)):\n                raise ValueError(f\'Invalid scaler: {k}\')\n        ratio = self.cfg[\'fusion_history_s\']/self.cfg[\'window_s\']\n        if ratio != int(ratio):\n            raise ValueError(\'Fusion history must contain whole windows.\')\n        self.history = int(ratio)\n        self.subject_index = {s:i for i,s in enumerate(sorted({r[\'subject\'] for r in self.inventory}))}\n        self._metadata = {}\n\n    def records(self, role):\n        if role not in self.roles:\n            raise ValueError(f\'{role} is unavailable in {self.scope}; allowed: {list(self.roles)}\')\n        rows = [r for r in self.inventory if r[\'subject\'] in self.roles[role]\n                and r[\'signal_ok\'] and not r[\'excluded_case\'] and r[\'split\']==\'pool\']\n        return sorted(rows, key=lambda r:(r[\'subject\'],r[\'case\'],r[\'start_s\'],r[\'relative_path\']))\n\n    def path(self, row):\n        return self.run/\'processed\'/row[\'case\']/\'pool\'/(Path(row[\'relative_path\']).stem+\'.h5\')\n\n    def metadata(self, row):\n        p = self.path(row)\n        if p not in self._metadata:\n            with h5py.File(p,\'r\') as h:\n                if str(h.attrs[\'case\']) != row[\'case\'] or json.loads(h.attrs[\'channels_json\']) != self.cfg[\'channels\']:\n                    raise ValueError(f\'Unexpected case or channels: {p}\')\n                if json.loads(h.attrs[\'config_json\']) != self.cfg:\n                    raise ValueError(f\'Unexpected processing configuration: {p}\')\n                keys = [\'label\',\'valid_signal\',\'window_start_s\',\'decision_s\',\'pair_with_previous\']\n                m = {k:h[k][:] for k in keys}\n            n = len(m[\'label\']); w = self.cfg[\'window_s\']\n            if any(len(v)!=n for v in m.values()) or not np.isin(m[\'label\'],[-1,0,1,2]).all():\n                raise ValueError(f\'Invalid metadata: {p}\')\n            if n and (not np.allclose(m[\'decision_s\']-m[\'window_start_s\'],w) or\n                      np.any(np.diff(m[\'decision_s\'])<=0)):\n                raise ValueError(f\'Invalid timeline: {p}\')\n            # History ends before current window starts. Reset on every EDF boundary,\n            # missing interval, or invalid signal; never concatenate across files.\n            ready = np.zeros(n,dtype=bool)\n            streak = 0\n            for i in range(n):\n                if i and not np.isclose(m[\'window_start_s\'][i],m[\'decision_s\'][i-1],rtol=0,atol=1e-6):\n                    streak = 0\n                if m[\'valid_signal\'][i]:\n                    ready[i] = streak >= self.history\n                    streak += 1\n                else:\n                    streak = 0\n            m[\'prediction_ready\'] = ready\n            self._metadata[p] = m\n        return self._metadata[p]\n\n    def index(self, role, mode=\'supervised\'):\n        if mode not in [\'supervised\',\'pairs\',\'replay\']:\n            raise ValueError(mode)\n        if mode==\'pairs\' and role!=\'train\':\n            raise ValueError(\'Future-window pairs are restricted to training.\')\n        items=[]\n        for row in self.records(role):\n            m=self.metadata(row); n=len(m[\'label\'])\n            if mode==\'replay\': keep=np.ones(n,dtype=bool)\n            else:\n                keep=m[\'prediction_ready\'] & (m[\'label\']>=0)\n                if mode==\'pairs\':\n                    next_ok=np.zeros(n,dtype=bool)\n                    if n>1:\n                        next_ok[:-1]=m[\'valid_signal\'][1:] & m[\'pair_with_previous\'][1:]\n                    keep &= next_ok\n            items.extend((row,int(i)) for i in np.flatnonzero(keep))\n        return items\n\n    def read(self, row, i, mode=\'supervised\'):\n        if mode not in [\'supervised\',\'pairs\',\'replay\']: raise ValueError(mode)\n        m=self.metadata(row); ready=bool(m[\'prediction_ready\'][i])\n        if mode!=\'replay\' and (not ready or m[\'label\'][i]<0):\n            raise ValueError(\'Supervised sample lacks labels or causal history.\')\n        C=len(self.cfg[\'channels\']); L=int(self.cfg[\'fs\']*self.cfg[\'window_s\']); H=self.history\n        inputs={\'raw_t\':np.zeros((C,L),np.float32), \'spec_t\':np.zeros((C,5),np.float32),\n                \'raw_past\':np.zeros((H,C,L),np.float32), \'spec_past\':np.zeros((H,C,5),np.float32)}\n        def normal(raw,spec):\n            s=self.scaler\n            x=((raw-s[\'raw_mean\'][:,None])/s[\'raw_std\'][:,None]).astype(np.float32)\n            f=((spec-s[\'feature_mean\'])/s[\'feature_std\']).astype(np.float32)\n            if not np.isfinite(x).all() or not np.isfinite(f).all():\n                raise ValueError(\'Nonfinite normalized input\')\n            return x,f\n        auxiliary={}\n        with h5py.File(self.path(row),\'r\') as h:\n            # Unavailable replay rows retain timestamps but carry zero placeholders.\n            # The model must run ONLY when prediction_ready is true.\n            if ready:\n                inputs[\'raw_t\'],inputs[\'spec_t\']=normal(h[\'raw_uv\'][i],h[\'features\'][i])\n                inputs[\'raw_past\'],inputs[\'spec_past\']=normal(h[\'raw_uv\'][i-H:i],h[\'features\'][i-H:i])\n            if mode==\'pairs\':\n                if row[\'subject\'] not in self.roles[\'train\']:\n                    raise ValueError(\'Future-window pairs are restricted to training subjects.\')\n                if i+1>=len(m[\'label\']) or not m[\'valid_signal\'][i+1] or not m[\'pair_with_previous\'][i+1]:\n                    raise ValueError(\'Invalid adjacent pair.\')\n                auxiliary[\'raw_tdt\'],auxiliary[\'spec_tdt\']=normal(h[\'raw_uv\'][i+1],h[\'features\'][i+1])\n        return dict(inputs=inputs, auxiliary=auxiliary,\n                    target=dict(label=int(m[\'label\'][i])),\n                    metadata=dict(subject=row[\'subject\'],case=row[\'case\'],patient_idx=self.subject_index[row[\'subject\']],\n                                  file=str(self.path(row)),index=i,start_s=float(m[\'window_start_s\'][i]),\n                                  decision_s=float(m[\'decision_s\'][i]),valid_signal=bool(m[\'valid_signal\'][i]),\n                                  prediction_ready=ready, reset_at_file_start=i==0))\n\n    def iterate(self, role, mode=\'replay\', shuffle=False, seed=20261028):\n        if shuffle and (role!=\'train\' or mode==\'replay\'):\n            raise ValueError(\'Only supervised training may be shuffled.\')\n        items=self.index(role,mode)\n        if shuffle:\n            rng=np.random.default_rng(seed); rng.shuffle(items)\n        for row,i in items: yield self.read(row,i,mode)\n\n    def audit(self, role):\n        rows=[];targets=[]\n        for row in self.records(role):\n            m=self.metadata(row)\n            rows.append(dict(subject=row[\'subject\'],file=row[\'relative_path\'],windows=len(m[\'label\']),\n                             valid_signal=int(m[\'valid_signal\'].sum()),prediction_ready=int(m[\'prediction_ready\'].sum()),\n                             **{f\'ready_label_{k}\':int((m[\'prediction_ready\']&(m[\'label\']==k)).sum()) for k in [-1,0,1,2]}))\n        for e in self.events:\n            if not e[\'eligible_target\'] or e[\'split\']!=\'pool\' or e[\'subject\'] not in self.roles[role]:continue\n            a=e[\'onset_s\']-self.cfg[\'sph_s\']-self.cfg[\'sop_s\'];b=e[\'onset_s\']-self.cfg[\'sph_s\']\n            n=0\n            for row in self.records(role):\n                if row[\'case\']!=e[\'case\']:continue\n                m=self.metadata(row)\n                # Decision times, independent of supervised labels.\n                n+=int((m[\'prediction_ready\']&(m[\'decision_s\']>=a)&(m[\'decision_s\']<=b)).sum())\n            targets.append(dict(event_id=e[\'event_id\'],subject=e[\'subject\'],retain_in_evaluation=True,\n                                available_forecast_decisions=n,has_forecast_opportunity=n>0))\n        return pd.DataFrame(rows),pd.DataFrame(targets)\n'
exec(compile(SOURCE, 'data_loading.py', "exec"))


In [ ]:
# Existing model_smoke.py
SOURCE = 'import numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass CoreMVModel(nn.Module):\n    """Explicit smoke-test foundation, not the complete physics-informed model."""\n    def __init__(self, channels=18, latent=32, rank=8):\n        super().__init__()\n        self.raw_conv=nn.Sequential(nn.Conv1d(channels,channels,7,padding=3,groups=channels),nn.GELU(),\n                                    nn.Conv1d(channels,64,1),nn.GELU())\n        self.raw_project=nn.Linear(128,latent)\n        self.spectral=nn.Sequential(nn.Flatten(1),nn.Linear(channels*5,64),nn.LayerNorm(64),nn.GELU(),nn.Linear(64,latent))\n        self.fusion=nn.Sequential(nn.Linear(latent*2,latent),nn.LayerNorm(latent),nn.GELU())\n        self.classifier=nn.Linear(latent,3)\n        self.U=nn.Parameter(torch.randn(latent,rank)*0.01)\n        self.V=nn.Parameter(torch.randn(latent,rank)*0.01)\n\n    def encode(self,raw,spec):\n        h=self.raw_conv(raw)\n        zr=self.raw_project(torch.cat([h.mean(-1),h.amax(-1)],dim=-1))\n        zs=self.spectral(spec)\n        return self.fusion(torch.cat([zr,zs],dim=-1))\n\n    def propagate(self,z):\n        # Row-vector convention for K=I+UV^T: z_next = z K^T.\n        return z+(z@self.V)@self.U.T\n\n    def forward(self,raw,spec):\n        z=self.encode(raw,spec)\n        return self.classifier(z),z\n\n    def operator_matrix(self):\n        return torch.eye(self.U.shape[0],device=self.U.device)+self.U@self.V.T\n\n\ndef tensor_batch(samples,device,paired=False):\n    def stack(group,key):\n        return torch.as_tensor(np.stack([s[group][key] for s in samples]),dtype=torch.float32,device=device)\n    out={\'raw\':stack(\'inputs\',\'raw_t\'),\'spec\':stack(\'inputs\',\'spec_t\'),\n         \'labels\':torch.tensor([s[\'target\'][\'label\'] for s in samples],dtype=torch.long,device=device)}\n    if paired:\n        out.update(next_raw=stack(\'auxiliary\',\'raw_tdt\'),next_spec=stack(\'auxiliary\',\'spec_tdt\'))\n    return out\n\n\ndef core_loss(model,batch,dynamics_weight=0.5):\n    logits,z=model(batch[\'raw\'],batch[\'spec\'])\n    ce=F.cross_entropy(logits,batch[\'labels\'])\n    with torch.no_grad():\n        target=model.encode(batch[\'next_raw\'],batch[\'next_spec\'])\n    dynamics=F.mse_loss(model.propagate(z),target)\n    return ce+dynamics_weight*dynamics,{\'classification\':ce,\'dynamics\':dynamics},z\n\n\ndef check_gradients(model):\n    report={}\n    for name,p in model.named_parameters():\n        if p.grad is None or not torch.isfinite(p.grad).all():\n            raise RuntimeError(f\'Missing/nonfinite gradient: {name}\')\n        report[name]=float(p.grad.norm().detach().cpu())\n    if not any(x>0 for x in report.values()): raise RuntimeError(\'All gradients are zero\')\n    return report\n\n\n@torch.no_grad()\ndef causal_probabilities(model, sample, device, top_k=8, encode_chunk=16):\n    if not sample[\'metadata\'][\'prediction_ready\']:\n        return None\n    inputs=sample[\'inputs\']\n    # Manuscript convention: last 74 preceding windows + current = 75.\n    # Loader requires 75 preceding valid windows, so its availability rule is\n    # conservatively one window stricter. Preserve it; never silently relax it.\n    H=len(inputs[\'raw_past\'])\n    if H<top_k or H<1: raise ValueError(\'Insufficient history for Top-K score\')\n    raw=np.concatenate([inputs[\'raw_past\'][1:],inputs[\'raw_t\'][None]],axis=0)\n    spec=np.concatenate([inputs[\'spec_past\'][1:],inputs[\'spec_t\'][None]],axis=0)\n    probabilities=[]\n    was_training=model.training;model.eval()\n    try:\n        for start in range(0,len(raw),encode_chunk):\n            logits,_=model(torch.as_tensor(raw[start:start+encode_chunk],device=device),\n                           torch.as_tensor(spec[start:start+encode_chunk],device=device))\n            probabilities.append(logits.softmax(-1).cpu())\n    finally:\n        model.train(was_training)\n    p=torch.cat(probabilities).numpy()\n    return dict(p_interictal=float(p[-1,0]),p_preictal=float(p[-1,1]),p_ictal=float(p[-1,2]),\n                top8_preictal=float(np.sort(p[:,1])[-top_k:].mean()))\n\n\ndef smoke_sample_indices(data,per_class=2,seed=20261028):\n    # Deliberately tiny class-stratified engineering test, not a training sampler.\n    candidates=data.index(\'train\',\'pairs\');rng=np.random.default_rng(seed)\n    picked=[]\n    for label in [0,1,2]:\n        eligible=[x for x in candidates if data.metadata(x[0])[\'label\'][x[1]]==label]\n        if not eligible: raise RuntimeError(f\'No class {label} dynamics pairs in this training split\')\n        chosen=rng.choice(len(eligible),size=min(per_class,len(eligible)),replace=False)\n        picked.extend(eligible[int(i)] for i in chosen)\n    return picked\n'
exec(compile(SOURCE, 'model_smoke.py', "exec"))


In [ ]:
# Existing physics_regularizer.py
SOURCE = 'import math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass WCInspired(nn.Module):\n    """16 uncoupled formal E/I pairs sharing 8 parameters; effective seconds.\n    Reduced sigmoid rate model, not the full original Wilson-Cowan equations.\n    """\n    def __init__(self):\n        super().__init__()\n        self.weight_logits=nn.Parameter(torch.logit(torch.tensor([1.5,1.,1.,0.5])/4))\n        self.tau_logits=nn.Parameter(torch.logit((torch.tensor([10.,16.])-4)/36))\n        self.drive_logits=nn.Parameter(torch.atanh(torch.tensor([-.5,-.5])/2))\n\n    def physical_parameters(self):\n        return 4*torch.sigmoid(self.weight_logits),4+36*torch.sigmoid(self.tau_logits),2*torch.tanh(self.drive_logits)\n\n    def activation(self,state):\n        E,I=state.chunk(2,dim=-1)\n        w,tau,b=self.physical_parameters()\n        return torch.cat([torch.sigmoid(w[0]*E-w[1]*I+b[0]),\n                          torch.sigmoid(w[2]*E-w[3]*I+b[1])],dim=-1)\n\n    def rhs(self,state):\n        _,tau,_=self.physical_parameters()\n        rate=torch.cat([tau[0].expand(state.shape[-1]//2),tau[1].expand(state.shape[-1]//2)])\n        return (self.activation(state)-state)/rate\n\n    def forward(self,state,dt_s=4.,substeps=16):\n        if state.shape[-1]!=32:raise ValueError(\'Expected 16 E + 16 I coordinates\')\n        if substeps<1 or int(substeps)!=substeps or not math.isfinite(dt_s) or dt_s<=0:\n            raise ValueError(\'Invalid integration settings\')\n        if not torch.isfinite(state).all() or torch.any(state<0) or torch.any(state>1):\n            raise ValueError(\'Initial state must be finite and in [0,1]\')\n        _,tau,_=self.physical_parameters()\n        rate=torch.cat([tau[0].expand(16),tau[1].expand(16)])\n        # Exponential Euler: exact leak, activation frozen per substep.\n        # Convex combination preserves [0,1] without clipping. This is NOT\n        # an assertion of global dynamical stability or numerical convergence.\n        decay=torch.exp(-(dt_s/int(substeps))/rate)\n        y=state\n        for _ in range(int(substeps)):\n            y=decay*y+(1-decay)*self.activation(y)\n        return y\n\nclass PhysicsMVModel(CoreMVModel):\n    def __init__(self,channels=18,latent=32,rank=8):\n        if latent!=32:raise ValueError(\'Physics v1 requires latent=32\')\n        super().__init__(channels,latent,rank)\n        self.wc=WCInspired()\n\n\ndef physics_loss(model,batch,dt_s=4.,substeps=16,dynamics_weight=.5,physics_weight=.2):\n    logits,z=model(batch[\'raw\'],batch[\'spec\'])\n    with torch.no_grad():\n        next_z=model.encode(batch[\'next_raw\'],batch[\'next_spec\'])\n        next_state=torch.sigmoid(next_z)\n    kz=model.propagate(z)\n    wc_state=model.wc(torch.sigmoid(z),dt_s=dt_s,substeps=substeps)\n    classification=F.cross_entropy(logits,batch[\'labels\'])\n    dynamics=F.mse_loss(kz,next_z)\n    coupling=F.mse_loss(torch.sigmoid(kz),wc_state)\n    empirical=F.mse_loss(wc_state,next_state)\n    physics=(coupling+empirical)/2\n    total=classification+dynamics_weight*dynamics+physics_weight*physics\n    return total,dict(classification=classification,dynamics=dynamics,\n                      physics_coupling=coupling,physics_empirical=empirical,physics=physics),z\n\n@torch.no_grad()\ndef physics_diagnostics(model,batch,dt_s=4.):\n    z=model.encode(batch[\'raw\'],batch[\'spec\'])\n    state=torch.sigmoid(z)\n    y16=model.wc(state,dt_s,16);y32=model.wc(state,dt_s,32);y256=model.wc(state,dt_s,256)\n    centered=z.double()-z.double().mean(0)\n    eigen=torch.linalg.svdvals(centered).square()\n    if eigen.sum()>1e-20:\n        p=eigen/eigen.sum();p=p[p>0];rank=float(torch.exp(-(p*p.log()).sum()).cpu())\n    else:rank=0.\n    K=model.operator_matrix().detach().double().cpu()\n    w,tau,b=model.wc.physical_parameters()\n    return dict(state_min=float(y16.min().cpu()),state_max=float(y16.max().cpu()),\n                max_difference_16_vs_32=float((y16-y32).abs().max().cpu()),\n                max_difference_16_vs_256=float((y16-y256).abs().max().cpu()),\n                max_difference_32_vs_256=float((y32-y256).abs().max().cpu()),\n                latent_mean_std=float(z.std(0,unbiased=False).mean().cpu()),\n                latent_effective_rank=rank,effective_rank_upper_bound=min(len(z)-1,z.shape[-1]),\n                state_saturated_fraction=float(((state<.01)|(state>.99)).float().mean().cpu()),\n                koopman_spectral_radius=float(torch.linalg.eigvals(K).abs().max()),\n                koopman_largest_singular_value=float(torch.linalg.svdvals(K).max()),\n                wc_weights=w.cpu().tolist(),wc_tau_effective_seconds=tau.cpu().tolist(),wc_drives=b.cpu().tolist())\n'
exec(compile(SOURCE, 'physics_regularizer.py', "exec"))


In [ ]:
"""Export ready-window fused latents and make training-fitted PCA figures.
Requires FoldData, CoreMVModel and PhysicsMVModel from the embedded source cells.
"""
from pathlib import Path
from contextlib import ExitStack
from datetime import datetime, timezone
import hashlib
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
import h5py
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()


def resolve_checkpoints(run, search_root, overrides, variants, expected):
    found = {}
    candidates = None
    for v in variants:
        wanted = expected[v]['model_checkpoint_sha256']
        if overrides.get(v):
            p = Path(overrides[v])
            if not p.is_file(): raise FileNotFoundError(p)
            if sha256(p) != wanted:
                raise ValueError(f'{v}: checkpoint hash differs from the selected seed/scope export. Do not mix model versions.')
            found[v] = p
            continue
        if candidates is None:
            print('Looking for last.pt under:', search_root, flush=True)
            candidates = sorted(Path(search_root).rglob('last.pt'))
        for p in candidates:
            if p.parent.name == v and sha256(p) == wanted:
                found[v] = p; break
        if v not in found:
            print(f'Found {len(candidates)} checkpoint files, but no matching {v} checkpoint.')
            for p in candidates:
                if p.parent.name == v: print('Candidate:', p)
            raise FileNotFoundError('Set SEARCH_ROOT to the folder containing your short-training experiment, or set CHECKPOINT_OVERRIDE to the exact file. No training was started.')
    return found


def load_selected_models(data, checkpoints, expected, device):
    models = {}
    for v,p in checkpoints.items():
        e = expected[v]
        if sha256(data.scaler_path) != e['scaler_sha256']:
            raise ValueError('Wrong training scaler for '+v)
        if sha256(data.run/'locked_folds.json') != e['folds_sha256']:
            raise ValueError('Locked folds differ from the selected prediction export')
        # These are your own checkpoints, verified against the supplied export hash.
        state = torch.load(p, map_location='cpu', weights_only=False)
        if state['variant'] != v or state['fingerprint'] != e['pilot_fingerprint']:
            raise ValueError('Checkpoint identity/fingerprint mismatch')
        if state['step'] != 10000:
            raise ValueError('Expected the completed 10,000-update checkpoint')
        cls = PhysicsMVModel if v == 'physics' else CoreMVModel
        model = cls(channels=len(data.cfg['channels']), latent=32, rank=8)
        model.load_state_dict(state['model'], strict=True)
        model.to(device).eval()
        models[v] = model
        print(v, 'loaded; step', state['step'], '; SHA-256', e['model_checkpoint_sha256'])
    return models


@torch.inference_mode()
def export_latents(data, models, expected, directory, device, batch_size=128):
    """Preserve ALL window rows; unready rows have NaN z and probabilities.
    The input reader uses no next-window observation, no labels and no history
    to compute z. Readiness matches the fixed forecasting availability policy.
    """
    directory = Path(directory); directory.mkdir(parents=True, exist_ok=True)
    paths = {}; summary = []
    for role in ['train','validation']:
        rows = data.records(role)
        (directory/f'events_{role}.json').write_text(json.dumps([e for e in data.events if e['subject'] in data.roles[role]],indent=2))
        with ExitStack() as stack:
            files = {}
            for v in models:
                path = directory/f'{v}_{role}_latents.h5'
                f = stack.enter_context(h5py.File(path,'x'))
                f.attrs.update(status='incomplete',role=role,variant=v,outer=data.outer,scope=data.scope,
                               checkpoint_sha256=expected[v]['model_checkpoint_sha256'],
                               scaler_sha256=expected[v]['scaler_sha256'],
                               folds_sha256=expected[v]['folds_sha256'],latent_dimension=32,
                               representation='current fused z before classifier; no temporal smoothing',
                               unavailable_policy='NaN z/probabilities for prediction_ready=False',
                               subjects_json=json.dumps(data.roles[role]),events_file=f'events_{role}.json')
                files[v] = f;paths[(v,role)] = path
            for ri,row in enumerate(rows):
                m = data.metadata(row);n = len(m['label']); groups = {}
                for v,f in files.items():
                    g = f.create_group(f'file_{ri:05d}')
                    g.attrs.update(subject=row['subject'],case=row['case'],relative_path=row['relative_path'])
                    for k in ['label','valid_signal','prediction_ready','window_start_s','decision_s']:
                        g.create_dataset(k,data=m[k],compression='gzip')
                    g.create_dataset('index',data=np.arange(n,dtype=np.int64),compression='gzip')
                    for key,width in [('z',32),('probabilities',3)]:
                        opts = dict(chunks=(min(batch_size,n),width),compression='gzip',compression_opts=1) if n else {}
                        g.create_dataset(key,shape=(n,width),dtype='f4',fillvalue=np.nan,**opts)
                    groups[v] = g
                with h5py.File(data.path(row),'r') as src:
                    for start in range(0,n,batch_size):
                        stop = min(n,start+batch_size)
                        keep = m['prediction_ready'][start:stop]
                        if not keep.any(): continue
                        ids = start + np.flatnonzero(keep)
                        s = data.scaler
                        raw = ((src['raw_uv'][start:stop][keep]-s['raw_mean'][:,None])/s['raw_std'][:,None]).astype('f4')
                        spec = ((src['features'][start:stop][keep]-s['feature_mean'])/s['feature_std']).astype('f4')
                        if not np.isfinite(raw).all() or not np.isfinite(spec).all():
                            raise ValueError('Nonfinite normalized input')
                        x = torch.as_tensor(raw,device=device);q = torch.as_tensor(spec,device=device)
                        for v,model in models.items():
                            logits,z = model(x,q)
                            zz = z.cpu().numpy(); pp = logits.softmax(-1).cpu().numpy()
                            if zz.shape != (len(ids),32) or not np.isfinite(zz).all() or not np.isfinite(pp).all():
                                raise ValueError('Invalid encoder output')
                            groups[v]['z'][ids] = zz
                            groups[v]['probabilities'][ids] = pp
                summary.append(dict(role=role,subject=row['subject'],file=row['relative_path'],windows=n,
                                    exported_latents=int(m['prediction_ready'].sum())))
                if ri % 10 == 0 or ri == len(rows)-1:
                    print(f'{role}: {ri+1}/{len(rows)} files; {row["relative_path"]}',flush=True)
            for f in files.values(): f.attrs['status'] = 'complete'
    pd.DataFrame(summary).to_csv(directory/'window_export_summary.csv',index=False)
    return paths


def sample_export(path, limit=50000, seed=20261101):
    """Uniform sample over exported ready windows using smallest random keys.
    Includes label -1 in the training PCA population. Bounded RAM.
    """
    rng = np.random.default_rng(seed);keys = np.empty(0);z = np.empty((0,32),dtype='f4');meta = pd.DataFrame()
    with h5py.File(path,'r') as f:
        if f.attrs['status'] != 'complete': raise ValueError('Incomplete latent export')
        role = f.attrs['role']
        for name in sorted(f):
            g = f[name]
            for start in range(0,len(g['label']),2048):
                stop = min(start+2048,len(g['label']));keep = g['prediction_ready'][start:stop].astype(bool)
                if not keep.any():continue
                zz = g['z'][start:stop][keep]
                if not np.isfinite(zz).all():raise ValueError('Missing exported ready latent')
                mm = pd.DataFrame(dict(subject=g.attrs['subject'],case=g.attrs['case'],file=g.attrs['relative_path'],role=role,
                                       index=g['index'][start:stop][keep],label=g['label'][start:stop][keep],
                                       decision_s=g['decision_s'][start:stop][keep]))
                keys = np.r_[keys,rng.random(len(zz))];z = np.concatenate([z,zz]);meta = pd.concat([meta,mm],ignore_index=True)
                if len(keys)>limit:
                    ids = np.argpartition(keys,limit-1)[:limit]
                    keys=keys[ids];z=z[ids];meta=meta.iloc[ids].reset_index(drop=True)
    if len(z)<3:raise ValueError('Insufficient exported windows for PCA')
    return z,meta


def fit_training_pca(train_path, limit, seed):
    with h5py.File(train_path,'r') as f:
        if f.attrs['role'] != 'train':raise ValueError('PCA fitting is restricted to the training role')
    z,meta = sample_export(train_path,limit,seed)
    if not np.any(np.ptp(z,axis=0)>0):
        raise ValueError('All sampled training latents are identical; PCA is undefined. Inspect representation collapse.')
    pca = PCA(n_components=2,svd_solver='full').fit(z)
    if not np.isfinite(pca.explained_variance_ratio_).all():
        raise ValueError('Undefined PCA variance; inspect the training representation.')
    return pca,z,meta


def save_figure(fig, directory, stem):
    fig.tight_layout(rect=[0,.06,1,.90])
    for ext in ['png','pdf','svg']:fig.savefig(Path(directory)/f'{stem}.{ext}',dpi=300,bbox_inches='tight')
    plt.show();plt.close(fig)


def pca_figures(paths, variants, directory, pca_limit=50000, plot_limit=12000, seed=20261101):
    plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'pdf.fonttype':42,'svg.fonttype':'none'})
    projectors = {}; report = []
    for v in variants:
        pca,ztrain,mtrain = fit_training_pca(paths[(v,'train')],pca_limit,seed)
        take = np.random.default_rng(seed+1).choice(len(ztrain),min(plot_limit,len(ztrain)),replace=False)
        ztrain = ztrain[take];mtrain=mtrain.iloc[take].reset_index(drop=True)
        zval,mval = sample_export(paths[(v,'validation')],plot_limit,seed+2)
        coords = [pca.transform(ztrain),pca.transform(zval)]
        projectors[v] = pca
        np.savez_compressed(Path(directory)/f'{v}_training_pca.npz',mean=pca.mean_,components=pca.components_,
                            explained_variance_ratio=pca.explained_variance_ratio_,fit_role='train',fit_windows=pca.n_samples_)
        for xy,frame,role in zip(coords,[mtrain,mval],['train','validation']):
            out = frame.assign(PC1=xy[:,0],PC2=xy[:,1]);out.to_csv(Path(directory)/f'{v}_{role}_pca_points.csv',index=False)
        fig,axes = plt.subplots(1,2,figsize=(11,4.8),sharex=True,sharey=True)
        colors = {0:'#7a97b2',1:'#d48200',2:'#b2182b'};names = {0:'Interictal',1:'Preictal',2:'Ictal'}
        for ax,xy,frame,title in zip(axes,coords,[mtrain,mval],['Training sample','Validation sample']):
            for label in [0,1,2]:
                mask=frame.label.to_numpy()==label
                ax.scatter(xy[mask,0],xy[mask,1],s=6 if label==0 else 11,alpha=.25 if label==0 else .65,
                           color=colors[label],label=f'{names[label]} (n={mask.sum():,})',rasterized=True)
            ax.set_title(title);ax.legend(frameon=False,fontsize=8)
            ax.set_xlabel(f'PC1 ({100*pca.explained_variance_ratio_[0]:.1f}% training variance)')
        axes[0].set_ylabel(f'PC2 ({100*pca.explained_variance_ratio_[1]:.1f}% training variance)')
        fig.suptitle(f'{v.capitalize()}: current fused latent representation\nOne PCA fitted on training windows; unchanged transformation for validation',fontsize=12)
        fig.text(.5,.01,'Uniform ready-window samples; excluded label −1 omitted from scatter only. Visual separation is not forecasting performance.',ha='center',fontsize=8)
        save_figure(fig,directory,f'{v}_pca_by_class')
        # Same validation points: class structure versus subject structure.
        xy=coords[1];fig,ax=plt.subplots(figsize=(8.5,5.2))
        subjects=sorted(mval.subject.unique());cm=plt.get_cmap('tab20')
        for i,subject in enumerate(subjects):
            mask=mval.subject.to_numpy()==subject
            ax.scatter(xy[mask,0],xy[mask,1],s=7,alpha=.4,color=cm(i%20),label=subject,rasterized=True)
        ax.legend(frameon=False,fontsize=8,ncol=2);ax.set_xlabel('PC1');ax.set_ylabel('PC2')
        fig.suptitle(f'{v.capitalize()}: validation latents colored by subject',fontsize=12)
        fig.text(.5,.01,'Training-fitted PCA; subject clustering can confound apparent class separation.',ha='center',fontsize=8)
        save_figure(fig,directory,f'{v}_pca_by_subject')
        report.append(dict(variant=v,pca_fit_role='train',fit_windows=int(pca.n_samples_),
                           train_plot_windows=len(ztrain),validation_plot_windows=len(zval),
                           explained_variance_ratio=pca.explained_variance_ratio_.tolist(),
                           train_label_counts=mtrain.label.value_counts().to_dict(),
                           validation_label_counts=mval.label.value_counts().to_dict()))
    (Path(directory)/'pca_report.json').write_text(json.dumps(report,indent=2))
    return projectors


def event_trajectory(data, paths, projectors, variants, directory):
    events = sorted([e for e in data.events if e['subject'] in data.roles['validation'] and e['eligible_target']],
                    key=lambda e:(e['case'],e['onset_s'],e['event_id']))
    if not events:
        print('No eligible validation event for a trajectory figure.');return
    event=events[0]  # Deterministic choice; do not choose an event by apparent separation/success.
    a,b=event['onset_s']-3600,event['onset_s']+300
    for v in variants:
        frames=[]
        with h5py.File(paths[(v,'validation')],'r') as f:
            for name in sorted(f):
                g=f[name]
                if g.attrs['case'] != event['case']:continue
                t=g['decision_s'][:];keep=(t>=a)&(t<=b)&g['prediction_ready'][:].astype(bool)
                if not keep.any():continue
                z=g['z'][:][keep];pc=projectors[v].transform(z)
                frames.append(pd.DataFrame(dict(file=g.attrs['relative_path'],decision_s=t[keep],PC1=pc[:,0],PC2=pc[:,1],
                                                 p_preictal=g['probabilities'][:][keep,1])))
        if not frames:
            print('No ready trajectory windows for',v,event['event_id']);continue
        frame=pd.concat(frames,ignore_index=True).sort_values('decision_s').reset_index(drop=True)
        frame['minutes_from_onset']=(frame.decision_s-event['onset_s'])/60
        frame.to_csv(Path(directory)/f'{v}_event_trajectory.csv',index=False)
        breaks=(frame.file!=frame.file.shift())|~np.isclose(frame.decision_s.diff(),data.cfg['window_s'],atol=1e-6,rtol=0)
        fig,axes=plt.subplots(3,1,figsize=(10,7),sharex=True)
        for _,part in frame.groupby(breaks.cumsum()):
            for ax,key in zip(axes,['PC1','PC2','p_preictal']):ax.plot(part.minutes_from_onset,part[key],color='#2166ac',lw=1)
        for ax in axes:
            ax.axvspan(-35,-5,color='#f5ca81',alpha=.3)
            ax.axvspan(0,(event['offset_s']-event['onset_s'])/60,color='#b2182b',alpha=.14)
            ax.axvline(0,color='#b2182b',ls='--',lw=1);ax.set_xlim(-60,5)
        axes[0].set_ylabel('PC1');axes[1].set_ylabel('PC2');axes[2].set_ylabel('Current preictal\nsoftmax score');axes[2].set_ylim(0,1)
        axes[2].set_xlabel('Minutes relative to seizure onset')
        fig.suptitle(f'{v.capitalize()}: {event["case"]}, {event["event_id"]}\nFirst eligible validation event by case/time; no selection by model outcome',fontsize=12)
        fig.text(.5,.01,'Orange: target preictal period; red: ictal interval. No lines connect EDF boundaries or unavailable intervals. Scores are unsmoothed.',ha='center',fontsize=8)
        save_figure(fig,directory,f'{v}_event_trajectory')
    (Path(directory)/'trajectory_event.json').write_text(json.dumps(event,indent=2))


def publish_outputs(local_dir, destination, manifest):
    """Copy finalized outputs once; avoid repeated small writes to Drive."""
    local_dir=Path(local_dir);destination=Path(destination)
    (local_dir/'manifest.json').write_text(json.dumps(manifest,indent=2))
    (local_dir/'README.txt').write_text(
        'The HDF5 files store one group per EDF and preserve every window row. z is the current 32-dimensional fused representation. '
        'Unready rows contain NaN z/probabilities; excluded supervised labels can still have valid ready latents. '
        'Class labels and annotations are metadata, never encoder inputs. probabilities are current unsmoothed class probabilities. '
        'PCA is fitted only on a uniform training-ready sample, including label -1. Validation uses the unchanged training projector. '
        'Class scatter plots omit label -1; subject plots retain it. Different models have separate PCA bases: coordinates are not directly aligned across models. '
        'No physiological identification, above-chance forecasting, or biological seizure boundary is established by these plots. '
        'A formal physics state, when needed, is sigmoid(z), with first 16 formal E and last 16 formal I; it is not measured physiology. '
        'The original EEG and model checkpoints are unchanged. No outer test data are accessed by this inner-fold export.\n')
    with zipfile.ZipFile(local_dir/'latent_figures_and_pca.zip','w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(local_dir.iterdir()):
            if p.is_file() and p.suffix not in ['.h5','.zip']:z.write(p,p.name)
    if destination.exists():raise FileExistsError('Use a new output directory: '+str(destination))
    shutil.copytree(local_dir,destination)
    print('Saved latent HDF5 files, PCA, figures and provenance to:',destination)
    return destination


In [ ]:
EXPECTED_CHECKPOINTS = {'20261028': {'inner_01': {'classifier': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_01', 'role': 'validation', 'variant': 'classifier', 'scaler_sha256': '530f4459c56e0c61361fa5ff3f6eddb719ae68cdc9252f240287db60dbd7f486', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': 'a420b76e24f4e9975c6ce26189bc5cd31e184949d80788e190a53c1818b3e4fb', 'pilot_fingerprint': 'b087b25cf0545578c0d5602bb9a042a967c039910a48f89c475d7a9b2984332c', 'smoothing': 'causal_mean_existing_history'}, 'koopman': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_01', 'role': 'validation', 'variant': 'koopman', 'scaler_sha256': '530f4459c56e0c61361fa5ff3f6eddb719ae68cdc9252f240287db60dbd7f486', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': '3f0bb19589912f97aaf9f9cae05c5aa894e41ebb51fef043fd2cc8040ce5635f', 'pilot_fingerprint': '20e40c3b51df09d70ed732c98f14bcc1120ec12f0c3fef7e9d114eb0e9be285d', 'smoothing': 'causal_mean_existing_history'}, 'physics': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_01', 'role': 'validation', 'variant': 'physics', 'scaler_sha256': '530f4459c56e0c61361fa5ff3f6eddb719ae68cdc9252f240287db60dbd7f486', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': '538b3b68f228ef08c90fcbfb08813d4d4223bd2377d03419931971f648639f30', 'pilot_fingerprint': '20e40c3b51df09d70ed732c98f14bcc1120ec12f0c3fef7e9d114eb0e9be285d', 'smoothing': 'causal_mean_existing_history'}}, 'inner_02': {'classifier': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_02', 'role': 'validation', 'variant': 'classifier', 'scaler_sha256': 'aaabbd3e2df3b76bb812a6b14f1abcff77e36f8061f1d16011613863d8b227d9', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': '6961bbad7b8562c6b572f90773972ee7a45f08e30f6fe0faf629e951ea30d68e', 'pilot_fingerprint': 'd1de13b390f3d2420901b8a78ba8fc069010cd46e9a10a2b37214db0bdad98ae', 'smoothing': 'causal_mean_existing_history'}, 'koopman': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_02', 'role': 'validation', 'variant': 'koopman', 'scaler_sha256': 'aaabbd3e2df3b76bb812a6b14f1abcff77e36f8061f1d16011613863d8b227d9', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': '974bbddd05d458e25fe37093e68b68b8ffdf960c5adf7762c6364371727a0e83', 'pilot_fingerprint': '4b00e5e113e310d3ce8cf4d439c4766bcf1f3538e85890b08f875821c370de59', 'smoothing': 'causal_mean_existing_history'}, 'physics': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_02', 'role': 'validation', 'variant': 'physics', 'scaler_sha256': 'aaabbd3e2df3b76bb812a6b14f1abcff77e36f8061f1d16011613863d8b227d9', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': '15a7ba1aeb1bf50c2b249e637eda88a9bfe0c72b274cf47ea67e4a1e095bb42a', 'pilot_fingerprint': '4b00e5e113e310d3ce8cf4d439c4766bcf1f3538e85890b08f875821c370de59', 'smoothing': 'causal_mean_existing_history'}}, 'inner_03': {'classifier': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_03', 'role': 'validation', 'variant': 'classifier', 'scaler_sha256': 'c7921e1f76911a05c6f9d89bcc6c1bbc2a899f7f56c3f2b3581e661946b66ee7', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': '29e0b231aac07858bcd84bd9d62bc5f5520d673426b9528159948fbd27ef3357', 'pilot_fingerprint': 'de8fbc64bc431111770a2cda5d8fc2d8fa26adb75dd6db74f1d1d39179ef30fd', 'smoothing': 'causal_mean_existing_history'}, 'koopman': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_03', 'role': 'validation', 'variant': 'koopman', 'scaler_sha256': 'c7921e1f76911a05c6f9d89bcc6c1bbc2a899f7f56c3f2b3581e661946b66ee7', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': 'f2df3832b13b2bc46f3c0b4a6b592e74f0de6c58a95580b928ce891e0c07060d', 'pilot_fingerprint': 'adbc38fa4007f71a220b7b82734bc3c97cbb9f1269de1c37062f5d367bff2839', 'smoothing': 'causal_mean_existing_history'}, 'physics': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_03', 'role': 'validation', 'variant': 'physics', 'scaler_sha256': 'c7921e1f76911a05c6f9d89bcc6c1bbc2a899f7f56c3f2b3581e661946b66ee7', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': 'cec39a087f81b0a19842f6d96ea9ff4d806cf7b6c6cb3d5b63995340e7c6f1ed', 'pilot_fingerprint': 'adbc38fa4007f71a220b7b82734bc3c97cbb9f1269de1c37062f5d367bff2839', 'smoothing': 'causal_mean_existing_history'}}}, '20261029': {'inner_01': {'classifier': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_01', 'role': 'validation', 'variant': 'classifier', 'scaler_sha256': '530f4459c56e0c61361fa5ff3f6eddb719ae68cdc9252f240287db60dbd7f486', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': '125c925fcd0e3f18d5f09f441e28fa87ae0d1a05a86d92c96a581e34dd023822', 'pilot_fingerprint': '480d805da6038410f2328a07f979de6140025c0af7430cd1c1c4c96c9491a727', 'smoothing': 'causal_mean_existing_history'}, 'koopman': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_01', 'role': 'validation', 'variant': 'koopman', 'scaler_sha256': '530f4459c56e0c61361fa5ff3f6eddb719ae68cdc9252f240287db60dbd7f486', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': 'dbd5ee08185dcfd0dc0a99515cfd86e4c71898d135f5e2827cbf8017b0dafea7', 'pilot_fingerprint': 'bdebe535f69d5766d5fd1d282efd25b4556c11fe41713287d6a265b081076c38', 'smoothing': 'causal_mean_existing_history'}, 'physics': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_01', 'role': 'validation', 'variant': 'physics', 'scaler_sha256': '530f4459c56e0c61361fa5ff3f6eddb719ae68cdc9252f240287db60dbd7f486', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': 'cc3822b47883ea4e7eeaf897f050c5794331e5cf9c3a691e477424a16b3b039b', 'pilot_fingerprint': 'bdebe535f69d5766d5fd1d282efd25b4556c11fe41713287d6a265b081076c38', 'smoothing': 'causal_mean_existing_history'}}, 'inner_02': {'classifier': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_02', 'role': 'validation', 'variant': 'classifier', 'scaler_sha256': 'aaabbd3e2df3b76bb812a6b14f1abcff77e36f8061f1d16011613863d8b227d9', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': '31475f2a226b936184deede970d13866994f79f325628b0ad0e16fa333ac9e2e', 'pilot_fingerprint': 'db03135c3ccdf9fc6dee4791d1b2bade3b02c104194ff944e9a50a933c26014d', 'smoothing': 'causal_mean_existing_history'}, 'koopman': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_02', 'role': 'validation', 'variant': 'koopman', 'scaler_sha256': 'aaabbd3e2df3b76bb812a6b14f1abcff77e36f8061f1d16011613863d8b227d9', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': '1d558801826c01fa9c489a05b35df1ea3c2c3641654613041d0439fda770d09b', 'pilot_fingerprint': '101c720bc6aaa50c09f5e6701a843b6c19755940be4e2bfe0969428c8ec65e16', 'smoothing': 'causal_mean_existing_history'}, 'physics': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_02', 'role': 'validation', 'variant': 'physics', 'scaler_sha256': 'aaabbd3e2df3b76bb812a6b14f1abcff77e36f8061f1d16011613863d8b227d9', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': '33b77d1e883a406807153ceb4dd10cbac6f47b774184ba3d204c323cba932ddf', 'pilot_fingerprint': '101c720bc6aaa50c09f5e6701a843b6c19755940be4e2bfe0969428c8ec65e16', 'smoothing': 'causal_mean_existing_history'}}, 'inner_03': {'classifier': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_03', 'role': 'validation', 'variant': 'classifier', 'scaler_sha256': 'c7921e1f76911a05c6f9d89bcc6c1bbc2a899f7f56c3f2b3581e661946b66ee7', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': '141c4f127bf259fe8493b1145b97d4907969abb95b0870191971e08f8dc10314', 'pilot_fingerprint': 'c62d3ba32663301db5312ecb9e8581c0adb5885b6c9c703f3b38cc3f29302bee', 'smoothing': 'causal_mean_existing_history'}, 'koopman': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_03', 'role': 'validation', 'variant': 'koopman', 'scaler_sha256': 'c7921e1f76911a05c6f9d89bcc6c1bbc2a899f7f56c3f2b3581e661946b66ee7', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': 'afa51f87be8c7647fa1ba56262ae689094d9e851f3bef727892ea994ae0c1c27', 'pilot_fingerprint': '555f0358003203c473492942835c6001eb2688d09cc1cfc28a2137836cfba377', 'smoothing': 'causal_mean_existing_history'}, 'physics': {'purpose': 'full_chronological_predictions', 'smoke_only': False, 'outer': 'outer_chb01', 'scope': 'inner_03', 'role': 'validation', 'variant': 'physics', 'scaler_sha256': 'c7921e1f76911a05c6f9d89bcc6c1bbc2a899f7f56c3f2b3581e661946b66ee7', 'folds_sha256': '76e6de77c12369879e6d2acd3b28bdfae4ec6e002df2267aa9df64269d99475b', 'model_checkpoint_sha256': 'fafc959d60d55c5b6ace943d716c9d6dc994a678a71c008f15d2295d911b0816', 'pilot_fingerprint': '555f0358003203c473492942835c6001eb2688d09cc1cfc28a2137836cfba377', 'smoothing': 'causal_mean_existing_history'}}}}


## 4. Load and verify the selected checkpoint

This checks the checkpoint SHA-256, variant, training fingerprint, 10,000-update step, scaler, and locked folds against your prediction-export provenance. If a matching checkpoint is not found, it stops rather than loading another experiment. Only your own hash-verified checkpoint is deserialized.

In [ ]:
if not RUN.is_dir():
    raise FileNotFoundError(f'Preprocessing run not found: {RUN}. Check the Drive mount and RUN.')
DATA = FoldData(RUN, outer=OUTER, scope=SCOPE)
assert DATA.cfg['fs'] == 256 and DATA.cfg['window_s'] == 4 and len(DATA.cfg['channels']) == 18
EXPECTED = EXPECTED_CHECKPOINTS[str(SEED)][SCOPE]
CHECKPOINTS = resolve_checkpoints(RUN, SEARCH_ROOT, CHECKPOINT_OVERRIDE, VARIANTS, EXPECTED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS = load_selected_models(DATA, CHECKPOINTS, EXPECTED, DEVICE)
# Engineering check only, using zero inputs. This does not fit anything.
with torch.inference_mode():
    for name,model in MODELS.items():
        raw = torch.zeros(2,18,1024,device=DEVICE)
        spec = torch.zeros(2,18,5,device=DEVICE)
        logits,z = model(raw,spec)
        assert z.shape == (2,32) and logits.shape == (2,3)
        torch.testing.assert_close(z,model.encode(raw,spec))
        torch.testing.assert_close(logits,model.classifier(z))
print('Device:',DEVICE)
print('Training subjects:',DATA.roles['train'])
print('Validation subjects:',DATA.roles['validation'])
print('No outer-test data will be exported.')


## 5. Export the representations

This reads the processed EEG and runs the encoder in batches. All requested models share the same input reads. Each HDF5 file retains every original window row; unavailable rows have NaN latent vectors. Ready label −1 windows are retained because they still have a valid representation, although they were excluded from supervised classification.

Exports are staged locally, then copied to Drive once in Section 7. No original files are overwritten. Progress prints every ten recordings.

In [ ]:
LATENT_PATHS = export_latents(DATA, MODELS, EXPECTED, LOCAL_DIR, DEVICE, batch_size=BATCH_SIZE)
for (variant,role),path in LATENT_PATHS.items():
    print(variant,role,path,round(path.stat().st_size/2**20,1),'MiB')


## 6. Training-fitted PCA and plots

Each model has one PCA fitted using its training-ready vectors. The same mean and components transform its validation vectors. Class labels and seizure annotations do not fit PCA.

The PCA plane retains only part of the 32-dimensional variation. Apparent clusters do not prove generalization or physiological identification. The subject-colored plot helps inspect whether class patterns may reflect subject differences. The trajectory uses the first eligible validation event by case/time, not a visually favorable or successfully forecast event.

In [ ]:
PROJECTORS = pca_figures(LATENT_PATHS, VARIANTS, LOCAL_DIR,
                         pca_limit=PCA_FIT_WINDOWS, plot_limit=PLOT_WINDOWS, seed=PLOT_SEED)
event_trajectory(DATA, LATENT_PATHS, PROJECTORS, VARIANTS, LOCAL_DIR)


## 7. Save everything to Drive

Each selected model produces two HDF5 files (train/validation), three figures in PNG/PDF/SVG, PCA components and point tables, and provenance. The smaller `latent_figures_and_pca.zip` contains plots and sampled PCA tables; full z arrays stay in the HDF5 files.

The checkpoint, EEG, thresholds, target eligibility, and original results are unchanged. This is descriptive representation analysis, not another training run.

In [ ]:
MANIFEST = dict(stage='latent_export_v1', outer=OUTER, scope=SCOPE, seed=SEED,
                variants=VARIANTS, run=str(RUN), representation='current fused z, 32 dimensions',
                checkpoint_paths={v:str(p) for v,p in CHECKPOINTS.items()},
                provenance={v:EXPECTED[v] for v in VARIANTS},
                training_subjects=DATA.roles['train'], validation_subjects=DATA.roles['validation'],
                roles_exported=['train','validation'], outer_test_accessed=False, retraining=False,
                pca_fit_role='train', pca_fit_window_limit=PCA_FIT_WINDOWS, plot_window_limit=PLOT_WINDOWS,
                plot_seed=PLOT_SEED, batch_size=BATCH_SIZE, device=str(DEVICE), torch_version=str(torch.__version__))
SAVED_TO = publish_outputs(LOCAL_DIR, DRIVE_DEST, MANIFEST)
print('Your latent representations are in:', SAVED_TO)


## 8. Reading an exported vector later

There is one group per EDF. To reconstruct PCA coordinates without refitting, use `(z - mean) @ components.T`. Probabilities are current unsmoothed class outputs, not the final smoothed alarm score. For the physics model, `sigmoid(z)` gives the formal bounded state; its E/I coordinate names do not identify measured physiological populations.

In [ ]:
# Inspect one exported ready-window representation.
VARIANT_TO_READ = VARIANTS[0]
with h5py.File(SAVED_TO / f'{VARIANT_TO_READ}_validation_latents.h5','r') as f:
    for file_id in sorted(f):
        g = f[file_id]
        ready_ids = np.flatnonzero(g['prediction_ready'][:])
        if len(ready_ids):
            i = int(ready_ids[0])
            print('Subject:',g.attrs['subject'],'; file:',g.attrs['relative_path'])
            print('Window:',i,'; decision time:',g['decision_s'][i],'; label:',g['label'][i])
            print('z shape:',g['z'][i].shape)
            print('z:',g['z'][i])
            break
